In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import SGDClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [31]:
df = pd.read_csv('./data/general/news.csv')
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (6335, 4)


,index,title,text,label
0,8476,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL


In [32]:
df['label'].value_counts()

label
REAL    3171
FAKE    3164
Name: count, dtype: int64

In [33]:
df['label'] = df['label'].map({'REAL': 0, 'FAKE': 1})

In [34]:
X, y = df['text'], df['label']

In [35]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [36]:
X_train_t, X_val, y_train_t, y_val = train_test_split(X_train, y_train, test_size=0.2)

In [37]:
X_train_t

1213    Posted on October 31, 2016 by Theodore Shoebat...
1128    Donald Trump's Problems Are Much Deeper Than A...
2610    Police Investigate Fraud after Voter Registrat...
4141    Iran took a hard stance on two of the biggest ...
237     Hillary Clinton has earned enough delegates to...
                              ...                        
2077    Neil Gabler has discounted political polarizat...
2128    News, information and analysis from the black ...
5517    Republican presidential candidate Carly Fiorin...
1022    New bionic eye implant connects directly to br...
3045    It's amazing to go back and watch Bernie Sande...
Name: text, Length: 4054, dtype: object

In [38]:
vectorizer = TfidfVectorizer(stop_words='english', max_df=0.7)
tfidf_train = vectorizer.fit_transform(X_train)
tfidf_train_t = vectorizer.transform(X_train_t)
tfidf_val = vectorizer.transform(X_val)
tfidf_test = vectorizer.transform(X_test)

In [39]:
model = SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0, max_iter=50)

In [40]:
model.fit(tfidf_train_t, y_train_t)
y_pred_val = model.predict(tfidf_val)
print("Validation Accuracy:", accuracy_score(y_val, y_pred_val))
print(classification_report(y_val, y_pred_val))
print("Confusion Matrix:\n", confusion_matrix(y_val, y_pred_val))

Validation Accuracy: 0.9349112426035503
              precision    recall  f1-score   support

           0       0.95      0.93      0.94       526
           1       0.92      0.94      0.93       488

    accuracy                           0.93      1014
   macro avg       0.93      0.94      0.93      1014
weighted avg       0.94      0.93      0.93      1014

Confusion Matrix:
 [[488  38]
 [ 28 460]]


In [41]:
model.fit(tfidf_train, y_train)
y_pred_test = model.predict(tfidf_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred_test))
print(classification_report(y_test, y_pred_test))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_test))

Test Accuracy: 0.936069455406472
              precision    recall  f1-score   support

           0       0.95      0.92      0.93       630
           1       0.93      0.95      0.94       637

    accuracy                           0.94      1267
   macro avg       0.94      0.94      0.94      1267
weighted avg       0.94      0.94      0.94      1267

Confusion Matrix:
 [[581  49]
 [ 32 605]]


### Applying to other Data

In [42]:
df_other = pd.read_csv('./data/general/cleaned_fakenewsdata.csv')
df_other['label'] = df_other['label'].map({'REAL': 0, 'FAKE': 1})
df_other.head()

,text,label
0,""""""" Study in journal , Nature Communications :...",0
1,""""""" Zelensky told the council that he was ther...",0
2,""""""" Central bankers and bureaucrats are seizin...",0
3,""""""" Last updated on .From the section Swimming...",0
4,THE discussion in the Morning Star and elsewhe...,0


In [43]:
X_other, y_other = df_other['text'], df_other['label']
vectorizer_other = TfidfVectorizer(stop_words='english', max_df=0.7)
tfidf_other = vectorizer.transform(X_other)
y_pred_other = model.predict(tfidf_other)
print("Other Dataset Accuracy:", accuracy_score(y_other, y_pred_other))
print(classification_report(y_other, y_pred_other))
print("Confusion Matrix:\n", confusion_matrix(y_other, y_pred_other))

Other Dataset Accuracy: 0.4700935561862714
              precision    recall  f1-score   support

           0       0.74      0.39      0.51      6790
           1       0.30      0.66      0.42      2723

    accuracy                           0.47      9513
   macro avg       0.52      0.53      0.47      9513
weighted avg       0.62      0.47      0.49      9513

Confusion Matrix:
 [[2669 4121]
 [ 920 1803]]


### Let's try a mixed model?

In [44]:
import pandas as pd

target = 2723
seed = 42  # or None for different random result each run

real = df_other[df_other['label'] == 0]
non_real = df_other[df_other['label'] != 0]

if len(real) > target:
    real = real.sample(n=target, random_state=seed)
# else: leave real as-is if already <= target

df_other = pd.concat([real, non_real], ignore_index=True)
df_other = df_other.sample(frac=1, random_state=seed).reset_index(drop=True)
df_other['label'].value_counts()

label
0    2723
1    2723
Name: count, dtype: int64

In [45]:
df_tot = pd.concat([df, df_other], ignore_index=True)
df_tot.drop(columns=['index', 'title'], inplace=True)
print("Combined Dataset shape:", df_tot.shape)
df_tot.head()

Combined Dataset shape: (11781, 2)


,text,label
0,"Daniel Greenfield, a Shillman Journalism Fello...",1
1,Google Pinterest Digg Linkedin Reddit Stumbleu...,1
2,U.S. Secretary of State John F. Kerry said Mon...,0
3,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",1
4,It's primary day in New York and front-runners...,0


In [46]:
X_tot, y_tot = df_tot['text'], df_tot['label']
X_train_tot, X_test_tot, y_train_tot, y_test_tot = train_test_split(X_tot, y_tot, test_size=0.2)
X_train_tot_t, X_val_tot, y_train_tot_t, y_val_tot = train_test_split(X_train_tot, y_train_tot, test_size=0.2)
tfidf_train_tot = vectorizer.fit_transform(X_train_tot)
tfidf_train_tot_t = vectorizer.transform(X_train_tot_t)
tfidf_val_tot = vectorizer.transform(X_val_tot)
tfidf_test_tot = vectorizer.transform(X_test_tot)

model = SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0, max_iter=50, class_weight='balanced')

model.fit(tfidf_train_tot_t, y_train_tot_t)
y_pred_val_tot = model.predict(tfidf_val_tot)
print("Combined Validation Accuracy:", accuracy_score(y_val_tot, y_pred_val_tot))
print(classification_report(y_val_tot, y_pred_val_tot))
print("Confusion Matrix:\n", confusion_matrix(y_val_tot, y_pred_val_tot))

Combined Validation Accuracy: 0.8026525198938992
              precision    recall  f1-score   support

           0       0.82      0.79      0.80       960
           1       0.79      0.82      0.80       925

    accuracy                           0.80      1885
   macro avg       0.80      0.80      0.80      1885
weighted avg       0.80      0.80      0.80      1885

Confusion Matrix:
 [[756 204]
 [168 757]]
